<a href="https://colab.research.google.com/github/edsilva14/infovis/blob/main/data360_exploration.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Exploración Profunda: API Data360 del Banco Mundial

Este cuaderno profundiza en la estructura de la API **Data360**, analizando los metadatos disponibles: cobertura temporal, áreas geográficas y categorías de datos (temas).

**API Endpoint**: `https://data360api.worldbank.org`  
**Documentación**: Data360 Open API Spec

In [ ]:
import requests
import pandas as pd
import duckdb
import altair as alt
import json

# Configuración base
BASE_URL = "https://data360api.worldbank.org/data360"
# alt.renderers.enable('colab') # Descomentar si se usa en Google Colabfrom sklearn.impute import SimpleImputer


Primero exploraremos las bases de datos disponibles en la API del Banco Mundial, sus indicadores y como se estructuran

In [ ]:
import requests
import pandas as pd

BASE_URL = "https://api.worldbank.org/v2"

print("1. Ver todas las bases de datos disponibles")
r = requests.get(f"{BASE_URL}/sources?format=json&per_page=100")
print("Status:", r.status_code)
data = r.json()
fuentes = data[1]  # la API WB devuelve [metadata, datos]
df_fuentes = pd.DataFrame(fuentes)[["id", "name"]]
print(df_fuentes)
print("2. Ver indicadores disponibles (con búsqueda por tema)")
r = requests.get(f"{BASE_URL}/indicator?format=json&per_page=50&source=2")
data = r.json()
indicadores = data[1]
df_ind = pd.DataFrame(indicadores)[["id", "name"]]
print(df_ind)
print("3. Traer datos concretos (población ARG, BRA, CHL,etc)")
url = "https://api.worldbank.org/v2/country/ARG;BRA;CHL/indicator/SP.POP.TOTL"

r = requests.get(url, params={
    "format": "json",
    "per_page": "500",
    "date": "2000:2023"
})

print("Status:", r.status_code)
print("URL consultada:", r.url)  # para ver exactamente qué se pidió
print("Respuesta cruda:", r.text[:300])

1. Ver todas las bases de datos disponibles
Status: 200
    id                                        name
0    1                              Doing Business
1    2                World Development Indicators
2    3             Worldwide Governance Indicators
3    5           Subnational Malnutrition Database
4    6               International Debt Statistics
..  ..                                         ...
66  89  Identification for Development (ID4D) Data
67  90                                    ICP 2021
68  91                                  PEFA_CRPFM
69  92                   Disability Data Hub (DDH)
70  93                         FPN Datahub Archive

[71 rows x 2 columns]
2. Ver indicadores disponibles (con búsqueda por tema
                      id                                               name
0      AG.CON.FERT.PT.ZS  Fertilizer consumption (% of fertilizer produc...
1         AG.CON.FERT.ZS  Fertilizer consumption (kilograms per hectare ...
2         AG.LND.AGRI.K2   

La API del Banco Mundial cuenta con un total de 71 bases de datos distintas.
La más completas de estas es la World Development Indicators (WDI, id=2) con más de 1.400 indicadores. Cada dato se estructura en 4 dimensiones:

País — Código ISO2 (AR) e ISO3 (ARG).

Tiempo — Año (series desde 1960 hasta 2024).

Indicador —Código único (ej: SP.POP.TOTL) + nombre descriptivo.

Valor — Número con unidad (%, USD, km², etc.)

## 1. Cobertura Temporal

¿Qué años de información podemos extraer? Analizaremos el rango temporal disponible para un indicador global representativo.

In [ ]:
def get_time_range(indicator_id="WB_WDI_SP_POP_TOTL", area="WLD"):
    url = f"{BASE_URL}/data"
    params = {"DATABASE_ID": "WB_WDI", "INDICATOR": indicator_id, "REF_AREA": area}
    resp = requests.get(url, params=params).json()
    years = [int(v['TIME_PERIOD']) for v in resp['value'] if v['TIME_PERIOD'].isdigit()]
    return min(years), max(years), len(years)

min_y, max_y, total_y = get_time_range()
print(f"Rango temporal disponible: {min_y} - {max_y} ({total_y} años)")

Rango temporal disponible: 1960 - 2024 (65 años)


## 2. Cobertura Geográfica (Países y Áreas)

La API no solo incluye países, sino también agregados regionales (ej. América Latina, Europa) y niveles de ingreso (ej. Países de ingreso bajo).

In [ ]:
def get_available_areas(indicator_id="WB_WDI_SP_POP_TOTL"):
    url = f"{BASE_URL}/data"
    params = {"DATABASE_ID": "WB_WDI", "INDICATOR": indicator_id, "isLatestData": True}
    resp = requests.get(url, params=params).json()
    areas = [v['REF_AREA'] for v in resp['value']]
    return sorted(list(set(areas)))

areas = get_available_areas()
print(f"Total de áreas/países disponibles: {len(areas)}")
print("Muestra de códigos de área (ISO3):", areas[:25])

Total de áreas/países disponibles: 265
Muestra de códigos de área (ISO3): ['ABW', 'AFE', 'AFG', 'AFW', 'AGO', 'ALB', 'AND', 'ARB', 'ARE', 'ARG', 'ARM', 'ASM', 'ATG', 'AUS', 'AUT', 'AZE', 'BDI', 'BEL', 'BEN', 'BFA', 'BGD', 'BGR', 'BHR', 'BHS', 'BIH']


In [ ]:
def get_all_area_metadata():
    """Obtiene metadatos extendidos (nombre, region) de todos los países usando la API de datos abiertos del Banco Mundial"""
    print("Obteniendo metadatos de áreas (regiones, nombres completos)...")
    url = "https://api.worldbank.org/v2/country"
    params = {"format": "json", "per_page": 300}
    try:
        resp = requests.get(url, params=params).json()
        # El formato es [metadata, [datos]]
        countries_data = resp[1]
        metadata_dict = {}
        for c in countries_data:
            metadata_dict[c['id']] = {
                'full_name': c['name'],
                'region': c['region']['value'] if isinstance(c['region'], dict) else 'Unknown',
                'income_level': c['incomeLevel']['value'] if isinstance(c['incomeLevel'], dict) else 'Unknown'
            }
        return metadata_dict
    except Exception as e:
        print(f"Error al obtener metadatos: {e}")
        return {}

area_metadata = get_all_area_metadata()
print(f"Metadatos cargados para {len(area_metadata)} áreas.")

Obteniendo metadatos de áreas (regiones, nombres completos)...
Metadatos cargados para 296 áreas.


## 3. Categorías de Datos (Temas)

Los indicadores están agrupados por temas. Utilizaremos el endpoint de metadatos para descubrir las categorías principales.

In [ ]:
def get_topics(top=200):
    url = f"{BASE_URL}/metadata"
    query = {"query": f"&$filter=series_description/database_id eq 'WB_WDI'&$select=series_description/topics/name&$top={top}"}
    resp = requests.post(url, json=query).json()

    unique_topics = set()
    if 'value' in resp:
        for item in resp['value']:
            sd = item.get('series_description')
            if sd is None:
                continue # Skip items where series_description is None

            # La API puede devolver series_description como dict o list segun el dataset
            # Ensure entries contains only dictionaries, filtering out any None or other types
            if isinstance(sd, list):
                entries = [e for e in sd if isinstance(e, dict)]
            elif isinstance(sd, dict):
                entries = [sd]
            else:
                entries = [] # Handle unexpected type for sd, skip if not dict or list

            for entry in entries:
                # entry is guaranteed to be a dict here
                for t in entry.get('topics', []):
                    # Ensure t is a dictionary before calling .get()
                    if isinstance(t, dict) and t.get('name'):
                        unique_topics.add(t['name'])
    return sorted(list(unique_topics))

topics = get_topics()
print(f"Temas principales detectados ({len(topics)}):")
for t in topics: print(f"- {t}")

Temas principales detectados (42):
- Agriculture and Food
- Carbon Pricing, Climate Finance and GHG accounting
- Climate Change
- Connectivity
- Crisis and Disaster Risk Finance
- Digital
- Digital Services
- Economic Policy
- Education
- Energy & Extractives
- Environment
- Finance
- Food and Nutrition Security
- Gender
- Global Infrastructure Finance
- Growth
- Growth and Jobs
- Health, Nutrition & Population
- Housing
- Inequality and Shared Prosperity
- Infrastructure
- Institutions
- Investment and Business Climate
- Jobs
- Learning
- Mortality
- People
- Planet
- Poverty
- Prosperity
- Schooling
- Social Protection
- Strengthening Agi Food Systems
- Teaching
- Trade Outcomes
- Trade, Investment and Competitiveness
- Transmission and Distribution
- Transport
- Trends in the Determinants of Food Security Outcomes
- Urban, Resilience and Land
- Water
- Water and the Economy


## 4. Visualización Exploratoria

### Crecimiento Comparativo en el Cono Sur
Utilizando los datos descubiertos, comparamos el crecimiento poblacional relativo entre países vecinos.

In [ ]:
def get_indicator_data(indicator, countries):
    url = f"{BASE_URL}/data"
    params = {"DATABASE_ID": "WB_WDI", "INDICATOR": indicator, "REF_AREA": ",".join(countries)}
    resp = requests.get(url, params=params).json()
    df = pd.DataFrame(resp['value'])
    df['OBS_VALUE'] = pd.to_numeric(df['OBS_VALUE'])
    df['TIME_PERIOD'] = pd.to_numeric(df['TIME_PERIOD'])
    return df

df_cono_sur = get_indicator_data("WB_WDI_SP_POP_TOTL", ["ARG", "CHL", "URY", "BRA"])

# Uso de DuckDB para filtrar o procesar si fuera necesario
df_filtered = duckdb.query("SELECT * FROM df_cono_sur WHERE TIME_PERIOD >= 1960").df()

# Visualización interactiva con Altair
line_chart = alt.Chart(df_filtered).mark_line(point=True).encode(
    x=alt.X('TIME_PERIOD:O', title='Año'),
    y=alt.Y('OBS_VALUE:Q', title='Población', scale=alt.Scale(type='log')),
    color=alt.Color('REF_AREA:N', title='País'),
    tooltip=['REF_AREA', 'TIME_PERIOD', 'OBS_VALUE']
).properties(
    title='Población Total en el Cono Sur (1960-2023)',
    width=600,
    height=300
).interactive()

line_chart

alt.Chart(...)

Este grafico se puede observar el crecimiento de la demografía a nivel latinoamericano, siendo Brasil y Argentina los principales contribuyentes al crecimiento poblacional en la región, seguido por Chile y Uruguay.

### 5. Preparación de Datos para PCA: Agrupando Países por Indicadores

Para realizar un PCA y visualizar países en un plano de dos dimensiones, necesitamos un conjunto de indicadores diversos para varios países. Seleccionaremos algunos indicadores representativos y un grupo más amplio de países para este análisis.

In [ ]:
# Definir una lista de indicadores relevantes para PCA
# Usamos algunos de los que ya hemos visto y añadimos otros para mayor diversidad
indicators_for_pca = {
    "WB_WDI_SP_POP_TOTL": "Poblacion Total", # Population, total
    "WB_WDI_NY_GDP_PCAP_CD": "PIB per Capita (USD)", # GDP per capita (current US$)
    "WB_WDI_SP_DYN_LE00_IN": "Esperanza de Vida al Nacer", # Life expectancy at birth, total (years)
    "WB_WDI_SE_ADT_LITR_ZS": "Tasa de Alfabetizacion", # Literacy rate, adult total (% of people ages 15 and above)
    "WB_WDI_EN_ATM_CO2E_PC": "Emisiones CO2 per Capita", # CO2 emissions (metric tons per capita)
    "WB_WDI_SH_MED_BEDS_ZS": "Camas Hospitalarias (per 1k personas)", # Hospital beds (per 1,000 people)
    "WB_WDI_IT_NET_USER_ZS": "Usuarios de Internet (% poblacion)" # Internet users (% of population)
}

# Seleccionar un conjunto de países para el análisis (ej. todos los de América Latina y el Caribe, y algunos de otras regiones)
# Usaremos los códigos ISO3 disponibles en la API
countries_for_pca = areas

# Funcion para obtener datos de un solo indicador para multiples areas y un año específico
def get_indicator_data_for_year(indicator_id, areas, year=None):
    url = f"{BASE_URL}/data"
    params = {"DATABASE_ID": "WB_WDI", "INDICATOR": indicator_id, "REF_AREA": ",".join(areas)}
    if year: # Intentar obtener el dato del año especificado, o el más cercano si no hay
        params['TIME_PERIOD'] = str(year)
        # params['isLatestData'] = True # Esto tomaría el último disponible si no hay para el año exacto
    else:
        params['isLatestData'] = True # Si no se especifica año, tomar el más reciente disponible

    resp = requests.get(url, params=params).json()
    df = pd.DataFrame(resp.get('value', []))

    # Asegurarse de que REF_AREA y OBS_VALUE estén presentes y limpios
    if df.empty:
        return pd.DataFrame()

    df['OBS_VALUE'] = pd.to_numeric(df['OBS_VALUE'], errors='coerce')
    df = df.dropna(subset=['OBS_VALUE'])

    # Si hay múltiples entradas por país (ej. diferentes años si no se usó isLatestData o un año específico)
    # nos quedamos con el valor más reciente o el que más se ajuste al año solicitado
    if 'TIME_PERIOD' in df.columns and not year: # Si no se pidió un año específico y se usó isLatestData
        df['TIME_PERIOD'] = pd.to_numeric(df['TIME_PERIOD'], errors='coerce')
        df = df.sort_values(by='TIME_PERIOD', ascending=False).drop_duplicates(subset=['REF_AREA'])
    elif 'TIME_PERIOD' in df.columns and year: # Si se pidió un año específico
        df['TIME_PERIOD'] = pd.to_numeric(df['TIME_PERIOD'], errors='coerce')
        # Intentar obtener el dato del año exacto, si no, el más cercano posterior, luego el más cercano anterior
        df['time_diff'] = abs(df['TIME_PERIOD'] - year)
        df = df.sort_values(by='time_diff').drop_duplicates(subset=['REF_AREA'])

    return df[['REF_AREA', 'OBS_VALUE']].rename(columns={'OBS_VALUE': indicator_id})

# Recopilar los datos para PCA
df_pca_input = pd.DataFrame({'REF_AREA': countries_for_pca})

for indicator_code, indicator_name in indicators_for_pca.items():
    print(f"Obteniendo datos para: {indicator_name} ({indicator_code})...")
    temp_df = get_indicator_data_for_year(indicator_code, countries_for_pca, year=2022) # Intentar obtener datos de 2022
    if not temp_df.empty:
        df_pca_input = pd.merge(df_pca_input, temp_df, on='REF_AREA', how='left')
    else:
        print(f"  No se encontraron datos para {indicator_name} en 2022. Intentando con datos más recientes...")
        temp_df = get_indicator_data_for_year(indicator_code, countries_for_pca) # Obtener el más reciente si 2022 no funciona
        if not temp_df.empty:
             df_pca_input = pd.merge(df_pca_input, temp_df, on='REF_AREA', how='left')
        else:
            print(f"  No se encontraron datos recientes para {indicator_name}.")

# Establecer REF_AREA como índice para facilitar el PCA
df_pca_input = df_pca_input.set_index('REF_AREA')

# Mostrar las primeras filas del DataFrame listo para PCA
print("\nDataFrame preparado para PCA:")
display(df_pca_input.head())

Obteniendo datos para: Poblacion Total (WB_WDI_SP_POP_TOTL)...
Obteniendo datos para: PIB per Capita (USD) (WB_WDI_NY_GDP_PCAP_CD)...
Obteniendo datos para: Esperanza de Vida al Nacer (WB_WDI_SP_DYN_LE00_IN)...
Obteniendo datos para: Tasa de Alfabetizacion (WB_WDI_SE_ADT_LITR_ZS)...
Obteniendo datos para: Emisiones CO2 per Capita (WB_WDI_EN_ATM_CO2E_PC)...
  No se encontraron datos para Emisiones CO2 per Capita en 2022. Intentando con datos más recientes...
  No se encontraron datos recientes para Emisiones CO2 per Capita.
Obteniendo datos para: Camas Hospitalarias (per 1k personas) (WB_WDI_SH_MED_BEDS_ZS)...
Obteniendo datos para: Usuarios de Internet (% poblacion) (WB_WDI_IT_NET_USER_ZS)...

DataFrame preparado para PCA:


,WB_WDI_SP_POP_TOTL,WB_WDI_NY_GDP_PCAP_CD,WB_WDI_SP_DYN_LE00_IN,WB_WDI_SE_ADT_LITR_ZS,WB_WDI_SH_MED_BEDS_ZS,WB_WDI_IT_NET_USER_ZS
REF_AREA,,,,,,
ABW,107310,30975.998912,76.226000,NaN,NaN,NaN
AFE,731821393,1679.327622,64.487020,73.055977,NaN,26.8000
AFG,40578842,357.261153,65.617000,NaN,0.36,17.1917
AFW,497387180,2138.473153,57.987813,60.780979,NaN,37.5000
AGO,35635029,3682.113151,64.246000,NaN,NaN,42.0719


In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer
import numpy as np

# 1. Preparación y Limpieza de Datos
# Tratar de mantener tantos países como sea posible
df_pca_cleaned = df_pca_input.copy()

# Eliminar indicadores (columnas) con demasiados valores nulos (> 50%)
thresh = len(df_pca_cleaned) * 0.5
df_pca_cleaned = df_pca_cleaned.dropna(axis=1, thresh=thresh)

print(f"Indicadores restantes después de filtrar por completitud: {df_pca_cleaned.shape[1]}")

# Imputar valores faltantes con la media de cada indicador
imputer = SimpleImputer(strategy='mean')
df_imputed = pd.DataFrame(imputer.fit_transform(df_pca_cleaned),
                          columns=df_pca_cleaned.columns,
                          index=df_pca_cleaned.index)

print(f"Realizando PCA sobre {df_imputed.shape[0]} áreas y {df_imputed.shape[1]} indicadores.")

if df_imputed.empty or df_imputed.shape[1] < 2:
    print("No hay suficientes datos para realizar PCA.")
else:
    # 2. Escalado y PCA
    scaler = StandardScaler()
    scaled_data = scaler.fit_transform(df_imputed)

    pca = PCA(n_components=2)
    principal_components = pca.fit_transform(scaled_data)

    pca_results = pd.DataFrame(data=principal_components, columns=['PC1', 'PC2'], index=df_imputed.index).reset_index()

    # 3. Enriquecimiento con Metadatos usando DuckDB
    # Convertimos el diccionario de metadatos a DataFrame para DuckDB
    meta_list = [{'REF_AREA': k, **v} for k, v in area_metadata.items()]
    df_meta = pd.DataFrame(meta_list)

    # Unimos resultados de PCA con Metadatos y Datos Originales para Tooltips
    original_data = df_imputed.reset_index()
    # Se filtran las agregaciones regionales (m.region != 'Aggregates') para analizar solo paises
    pca_final = duckdb.query("""
        SELECT
            p.REF_AREA,
            p.PC1,
            p.PC2,
            m.full_name as 'País',
            m.region as 'Región',
            o.* EXCLUDE (REF_AREA)
        FROM pca_results p
        LEFT JOIN df_meta m ON p.REF_AREA = m.REF_AREA
        JOIN original_data o ON p.REF_AREA = o.REF_AREA
        WHERE m.region IS NOT NULL AND m.region != 'Aggregates'
    """).df()

    # 4. Mapeo de nombres legibles para el tooltip
    friendly_tooltips = [
        alt.Tooltip('REF_AREA:N', title='Código ISO'),
        alt.Tooltip('País:N'),
        alt.Tooltip('Región:N'),
        alt.Tooltip('PC1:Q', format='.2f'),
        alt.Tooltip('PC2:Q', format='.2f')
    ]

    # Añadir indicadores originales con nombres legibles al tooltip
    for col in df_imputed.columns:
        friendly_name = indicators_for_pca.get(col, col)
        friendly_tooltips.append(alt.Tooltip(f'{col}:Q', title=friendly_name, format='.2f'))

    # 5. Visualización Interactiva con Altair
    selection = alt.selection_point(fields=['Región'], bind='legend')

    base = alt.Chart(pca_final).encode(
        x=alt.X('PC1:Q', title=f'PC1 ({pca.explained_variance_ratio_[0]*100:.2f}%)'),
        y=alt.Y('PC2:Q', title=f'PC2 ({pca.explained_variance_ratio_[1]*100:.2f}%)')
    )

    scatter = base.mark_circle(size=80, opacity=0.7).encode(
        color=alt.Color('Región:N', scale=alt.Scale(scheme='tableau10'), title='Región Geográfica'),
        opacity=alt.condition(selection, alt.value(0.9), alt.value(0.1)),
        tooltip=friendly_tooltips
    ).add_params(
        selection
    )

    # Representar codigo ISO del pais en el grafico
    text = base.mark_text(align='left', baseline='middle', dx=7, fontSize=10).encode(
        text='REF_AREA:N',
        color=alt.value('black'),
        opacity=alt.condition(selection, alt.value(0.8), alt.value(0.1))
    )

    chart = (scatter + text).properties(
        title={
            "text": "PCA de Países en el Año 2022 basado en Indicadores de Desarrollo",
            "subtitle": "Interactividad: Clic en la leyenda para filtrar, hover para ver detalles"
        },
        width=800,
        height=600
    ).interactive()

    display(chart)


Indicadores restantes después de filtrar por completitud: 4
Realizando PCA sobre 265 áreas y 4 indicadores.


alt.LayerChart(...)

### Siguientes Pasos: Escalado, PCA y Visualización

Ahora que tenemos el DataFrame `df_pca_input`, los siguientes pasos serían:

1.  **Manejo de Valores Faltantes**: Imputar o eliminar filas/columnas con datos faltantes. (Aunque la función `get_indicator_data_for_year` ya los maneja un poco).
2.  **Escalado de Datos**: Los indicadores tienen diferentes unidades y escalas, por lo que es crucial escalarlos (ej. con `StandardScaler` de `sklearn.preprocessing`) antes de aplicar PCA.
3.  **Aplicar PCA**: Utilizar `PCA` de `sklearn.decomposition` para reducir la dimensionalidad a 2 componentes principales.
4.  **Visualización**: Graficar los países en un diagrama de dispersión usando las dos primeras componentes principales, lo que permitirá identificar grupos de países con características similares.

## Resumen Técnico de la Exploración

- **Rango Temporal**: 1960 a 2024. Ideal para series de largo plazo.
- **Cobertura Espacial**: 265 códigos de área, permitiendo análisis país por país o por bloques económicos (G20, OCDE, etc.).
- **Temáticas**: Diversidad desde agricultura y medio ambiente hasta políticas económicas y agua.
- **Facilidad de Uso**: La API permite filtrado complejo mediante OData (usado en el endpoint de metadatos).

### Interpretación de los Componentes Principales (PCA)

Para entender por qué los países se agrupan de cierta manera en el gráfico interactivo, es fundamental analizar cómo influyen (qué "cargas" o pesos tienen) las variables originales en cada componente principal:

*   **Componente Principal 1 (PC1 - Nivel de Desarrollo Socioeconómico)**:
    Por lo general, el PC1 captura el nivel de desarrollo y riqueza general del país. Variables como el **PIB per Cápita**, la **Esperanza de Vida**, la **Tasa de Alfabetización** y los **Usuarios de Internet** tienen una influencia muy fuerte y positiva en este eje. Por otro lado, suele correlacionarse positivamente con mayores **Emisiones de CO2**. Países situados más a la derecha (PC1 alto) tienden a ser naciones desarrolladas, mientras que los situados a la izquierda enfrentan mayores desafíos socioeconómicos.

*   **Componente Principal 2 (PC2 - Dimensión Demográfica)**:
    El PC2 suele estar fuertemente dominado por variables de escala o tamaño absoluto, principalmente la **Población Total**. Países situados en los extremos de este eje se separan del montón exclusivamente por su masiva cantidad de habitantes (como India o China), diferenciándose independientemente de su nivel de riqueza.

Esta ortogonalidad permite que el gráfico evalúe de forma simultánea el progreso socioeconómico (eje X) y el peso demográfico (eje Y) de cada rincón del mundo.

In [ ]:
# 6. Tabla de Datos Tidy, Formateada y Filtrable
from IPython.display import display, HTML
import numpy as np

# Renombrar columnas a nombres legibles
df_tidy = pca_final.rename(columns=indicators_for_pca)

# Mantener solo columnas relevantes y ordenarlas lógicamente
cols_base = ['REF_AREA', 'País', 'Región', 'PC1', 'PC2']
# Extraer indicadores presentes
cols_indicadores = [indicators_for_pca.get(c, c) for c in original_data.columns if c not in ['REF_AREA', 'index']]
columnas_ordenadas = cols_base + [c for c in cols_indicadores if c in df_tidy.columns]
df_tidy = df_tidy[[c for c in columnas_ordenadas if c in df_tidy.columns]]

# Formato personalizado continuo para lograr una tabla 'Tidy'
# - Separador de miles con punto (.)
# - Coma decimal (,)
# - Simplificar > 10.000 como enteros sin decimales
def format_val(x):
    if pd.isna(x):
        return "Sin Datos"
    if isinstance(x, (int, float, np.number)):
        if abs(x) > 10000:
            return f"{x:,.0f}".replace(",", ".")
        else:
            return f"{x:,.2f}".replace(",", "X").replace(".", ",").replace("X", ".")
    return str(x)

df_formatted = df_tidy.copy()
for col in df_formatted.select_dtypes(include=[np.number]).columns:
    df_formatted[col] = df_formatted[col].apply(format_val)

# Convertir a HTML
html_table = df_formatted.to_html(index=False, table_id="tidy_table", classes="display")

# Añadir inyección para renderizar una tabla de datos altamente interactiva (DataTables JS)
html_code = f"""
<link rel="stylesheet" type="text/css" href="https://cdn.datatables.net/1.13.6/css/jquery.dataTables.min.css">
<script type="text/javascript" charset="utf8" src="https://code.jquery.com/jquery-3.7.0.min.js"></script>
<script type="text/javascript" charset="utf8" src="https://cdn.datatables.net/1.13.6/js/jquery.dataTables.min.js"></script>

{html_table}

<script>
// Evitar colisiones de librerías en múltiples re-ejecuciones
setTimeout(function() {{
    if ($.fn.dataTable.isDataTable('#tidy_table')) {{
        $('#tidy_table').DataTable().destroy();
    }}
    $('#tidy_table').DataTable({{
        "pageLength": 10,
        "lengthMenu": [5, 10, 25, 50, 100],
        "language": {{
            "url": "//cdn.datatables.net/plug-ins/1.13.6/i18n/es-ES.json"
        }}
    }});
}}, 100);
</script>
<style>
/* Estilo limpio (Tidy) para la tabla */
#tidy_table {{
    font-family: Arial, Helvetica, sans-serif;
    border-collapse: collapse;
    width: 100%;
    margin-top: 20px;
}}
#tidy_table td, #tidy_table th {{
    border: 1px solid #ddd;
    padding: 8px;
    text-align: right;
}}
#tidy_table th {{
    text-align: center;
    background-color: #f2f2f2;
    color: black;
}}
#tidy_table tr:hover {{background-color: #f9f9f9;}}
</style>
"""

display(HTML(html_code))


REF_AREA,País,Región,PC1,PC2,Poblacion Total,PIB per Capita (USD),Esperanza de Vida al Nacer,Usuarios de Internet (% poblacion)
ABW,Aruba,Latin America & Caribbean,"0,52","-0,28",107.310,30.976,"76,23","68,77"
AFG,Afghanistan,"Middle East, North Africa, Afghanistan & Pakistan","-2,24","-0,56",40.578.842,"357,26","65,62","17,19"
AGO,Angola,Sub-Saharan Africa,"-1,64","-0,50",35.635.029,"3.682,11","64,25","42,07"
ALB,Albania,Europe & Central Asia,"0,64","-0,16",2.451.636,"7.756,96","78,77","82,61"
AND,Andorra,Europe & Central Asia,"2,02","-0,08",79.705,42.414,"84,02","94,49"
ARE,United Arab Emirates,"Middle East, North Africa, Afghanistan & Pakistan","2,04","-0,12",10.074.977,50.760,"80,49","100,00"
ARG,Argentina,Latin America & Caribbean,"0,67","-0,15",45.407.904,13.962,"75,81","88,38"
ARM,Armenia,Europe & Central Asia,"0,16","-0,24",2.969.200,"6.571,97","74,77","77,03"
ASM,American Samoa,East Asia & Pacific,"0,00","-0,32",48.342,18.017,"72,75","68,77"
ATG,Antigua and Barbuda,Latin America & Caribbean,"0,63","-0,22",92.840,20.105,"77,48","77,16"


## 7. Gráfico de Barras Interactivo · Escala Logarítmica · Filtro por Región

Comparación de indicadores del Banco Mundial entre países de una misma región.  
**Controles:** seleccioná la región geográfica y la variable de interés.  
**Escala:** logarítmica (permite comparar países con órdenes de magnitud muy distintos).  
**Color:** nivel de ingreso (World Bank classification).

In [ ]:
import altair as alt
import pandas as pd
import requests

BASE_URL = "https://data360api.worldbank.org/data360"

# ── Indicadores a visualizar ──────────────────────────────────────────────────
INDICATORS = {
    "WB_WDI_NY_GDP_MKTP_CD":    "GDP Total (US$)",
    "WB_WDI_NY_GNP_MKTP_CD":    "GNI Total (US$)",
    "WB_WDI_SP_POP_TOTL":       "Población Total",
    "WB_WDI_SH_XPD_CHEX_PC_CD": "Gasto en Salud per cápita (US$)",
}

# ── Metadatos de países (nombre completo, región, nivel de ingreso) ────────────
def get_country_meta():
    url = "https://api.worldbank.org/v2/country"
    resp = requests.get(url, params={"format": "json", "per_page": 300}).json()
    rows = []
    for c in resp[1]:
        region = c["region"]["value"] if isinstance(c["region"], dict) else None
        if region and region != "Aggregates":
            rows.append({
                "REF_AREA":    c["id"],
                "country_name": c["name"],
                "region":      region,
                "income_level": c["incomeLevel"]["value"]
                               if isinstance(c["incomeLevel"], dict)
                               else "Not classified",
            })
    return pd.DataFrame(rows)

print("Cargando metadatos de países...")
df_meta_bar = get_country_meta()
country_codes_bar = df_meta_bar["REF_AREA"].tolist()
print(f"  {len(country_codes_bar)} países con metadatos")

# ── Función: último dato disponible para un indicador ────────────────────────
def fetch_latest(indicator_id, areas):
    url = f"{BASE_URL}/data"
    params = {
        "DATABASE_ID":  "WB_WDI",
        "INDICATOR":    indicator_id,
        "REF_AREA":     ",".join(areas),
        "isLatestData": "true",   # string lowercase: más compatible con la API
    }
    resp = requests.get(url, params=params).json()
    df = pd.DataFrame(resp.get("value", []))
    if df.empty:
        print(f"    ⚠ Sin datos para {indicator_id}")
        return pd.DataFrame(columns=["REF_AREA", "OBS_VALUE", "TIME_PERIOD"])
    df["OBS_VALUE"]   = pd.to_numeric(df["OBS_VALUE"],   errors="coerce")
    df["TIME_PERIOD"] = pd.to_numeric(df["TIME_PERIOD"], errors="coerce")
    df = df.sort_values("TIME_PERIOD", ascending=False).drop_duplicates("REF_AREA")
    return df[["REF_AREA", "OBS_VALUE", "TIME_PERIOD"]]

# ── Carga de todos los indicadores ────────────────────────────────────────────
frames = []
for ind_id, ind_name in INDICATORS.items():
    print(f"  Cargando: {ind_name}...")
    df_ind = fetch_latest(ind_id, country_codes_bar)
    df_ind["variable"] = ind_name
    frames.append(df_ind)

df_bar = pd.concat(frames, ignore_index=True)
df_bar = df_bar.merge(df_meta_bar, on="REF_AREA", how="left")
df_bar = df_bar.dropna(subset=["OBS_VALUE", "region"])
df_bar["year_label"] = df_bar["TIME_PERIOD"].astype(int).astype(str)

# ── Diagnóstico ───────────────────────────────────────────────────────────────
print(f"\n✓ Dataset listo: {len(df_bar):,} filas · "
      f"{df_bar['REF_AREA'].nunique()} países · "
      f"{df_bar['variable'].nunique()} variables")
print("\nRegiones disponibles:")
for r in sorted(df_bar["region"].unique()):
    n = df_bar[df_bar["region"] == r]["REF_AREA"].nunique()
    print(f"  {r!r}  ({n} países)")
print("\nVariables disponibles:", df_bar["variable"].unique().tolist())
print("\nPrimeras filas:")
display(df_bar.head(4))


Cargando metadatos de países...
  217 países con metadatos
  Cargando: GDP Total (US$)...
  Cargando: GNI Total (US$)...
  Cargando: Población Total...
  Cargando: Gasto en Salud per cápita (US$)...

✓ Dataset listo: 833 filas · 217 países · 4 variables

Regiones disponibles:
  'East Asia & Pacific'  (37 países)
  'Europe & Central Asia'  (58 países)
  'Latin America & Caribbean '  (42 países)
  'Middle East, North Africa, Afghanistan & Pakistan'  (23 países)
  'North America'  (3 países)
  'South Asia'  (6 países)
  'Sub-Saharan Africa '  (48 países)

Variables disponibles: ['GDP Total (US$)', 'GNI Total (US$)', 'Población Total', 'Gasto en Salud per cápita (US$)']

Primeras filas:


,REF_AREA,OBS_VALUE,TIME_PERIOD,variable,country_name,region,income_level,year_label
0,GUY,2.466271e+10,2024,GDP Total (US$),Guyana,Latin America & Caribbean,High income,2024
1,GNB,2.218394e+09,2024,GDP Total (US$),Guinea-Bissau,Sub-Saharan Africa,Low income,2024
2,HTI,2.522415e+10,2024,GDP Total (US$),Haiti,Latin America & Caribbean,Lower middle income,2024
3,HND,3.709357e+10,2024,GDP Total (US$),Honduras,Latin America & Caribbean,Lower middle income,2024


In [ ]:
# ── Desactivar límite de filas de Altair (necesario en Colab) ─────────────────
alt.data_transformers.disable_max_rows()

regions_bar   = sorted(df_bar["region"].unique().tolist())
variables_bar = list(INDICATORS.values())

INCOME_ORDER  = [
    "High income", "Upper middle income",
    "Lower middle income", "Low income", "Not classified",
]
INCOME_COLORS = ["#1a7abf", "#4cb4e7", "#f5a623", "#e05c2f", "#aaaaaa"]

# ── Selecciones bound a dropdowns (selection_point es más robusto en Colab) ──
input_region = alt.binding_select(options=regions_bar, name="Región: ")
region_sel = alt.selection_point(
    name="region_sel",
    fields=["region"],
    bind=input_region,
    value="Latin America & Caribbean",
)

input_var = alt.binding_select(options=variables_bar, name="Variable: ")
var_sel = alt.selection_point(
    name="var_sel",
    fields=["variable"],
    bind=input_var,
    value=variables_bar[0],
)

# ── Gráfico principal ─────────────────────────────────────────────────────────
bar_chart = (
    alt.Chart(df_bar)
    .mark_bar(cornerRadiusTopLeft=4, cornerRadiusTopRight=4)
    .encode(
        x=alt.X(
            "country_name:N",
            sort=alt.EncodingSortField(field="OBS_VALUE", order="descending"),
            title=None,
            axis=alt.Axis(labelAngle=-50, labelLimit=150, labelFontSize=10),
        ),
        y=alt.Y(
            "OBS_VALUE:Q",
            scale=alt.Scale(type="log"),
            title="Valor (escala logarítmica)",
            axis=alt.Axis(format="~s", grid=True),
        ),
        color=alt.Color(
            "income_level:N",
            title="Nivel de ingreso",
            sort=INCOME_ORDER,
            scale=alt.Scale(domain=INCOME_ORDER, range=INCOME_COLORS),
            legend=alt.Legend(orient="bottom", columns=3),
        ),
        opacity=alt.condition(
            region_sel & var_sel, alt.value(0.85), alt.value(0.0)
        ),
        tooltip=[
            alt.Tooltip("country_name:N",  title="País"),
            alt.Tooltip("region:N",        title="Región"),
            alt.Tooltip("variable:N",      title="Variable"),
            alt.Tooltip("OBS_VALUE:Q",     title="Valor", format=",.2~f"),
            alt.Tooltip("income_level:N",  title="Nivel de ingreso"),
            alt.Tooltip("year_label:N",    title="Año del dato"),
        ],
    )
    .transform_filter(region_sel)
    .transform_filter(var_sel)
    .add_params(region_sel, var_sel)
    .properties(
        title={
            "text": "Indicadores del Banco Mundial por País y Región",
            "subtitle": (
                "Seleccioná región y variable  ·  Escala logarítmica  ·  "
                "Fuente: World Bank Data360 API"
            ),
            "anchor": "start",
            "fontSize": 16,
            "subtitleFontSize": 12,
            "subtitleColor": "#666",
        },
        width=880,
        height=440,
    )
)

bar_chart


alt.Chart(...)